# Step 5: Modeling, Evaluasi, dan Perbandingan dengan Paper

Melatih 5 model klasifikasi (Decision Tree, SVM, MLP, Random Forest, XGBoost) dengan parameter yang dilaporkan paper, pada data hasil pipeline preprocessing terkoreksi.

Skenario yang direplikasi: kombinasi seleksi fitur chi-square (top-20) + SMOTE, yaitu kombinasi terbaik yang dilaporkan paper untuk XGBoost. Sesuai batasan brief Tugas #2, kita tidak mengulang seluruh 6 skenario ablation paper dan tidak melakukan tuning ulang -- fokus ke skenario terbaik yang jadi bahan perbandingan utama di laporan.

SMOTE hanya diterapkan pada data train, test tetap mencerminkan distribusi asli.

In [1]:
import sys, os
sys.path.append('../src')
from models import get_models
from evaluation import evaluate_model
import pandas as pd
import numpy as np
from imblearn.over_sampling import SMOTE

os.makedirs('../results/tables', exist_ok=True)

RANDOM_STATE = 42

X_train = pd.read_csv('../data/processed/X_train.csv')
X_test = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').iloc[:, 0]
y_test = pd.read_csv('../data/processed/y_test.csv').iloc[:, 0]

print('X_train:', X_train.shape, '| X_test:', X_test.shape)
print('Proporsi kelas positif -- train (sebelum SMOTE):', y_train.mean().round(4))
print('Proporsi kelas positif -- test (tidak di-resample):', y_test.mean().round(4))


X_train: (8631, 20) | X_test: (3699, 20)
Proporsi kelas positif -- train (sebelum SMOTE): 0.1548
Proporsi kelas positif -- test (tidak di-resample): 0.1546


## 1. SMOTE Oversampling (Hanya pada Data Train)

SMOTE cuma dipakai di train. Test tidak di-resample, biar evaluasi tetap merefleksikan kondisi nyata (imbalanced).

In [2]:
smote = SMOTE(random_state=RANDOM_STATE)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print('Sebelum SMOTE -- train:', X_train.shape, '| distribusi kelas:', y_train.value_counts().to_dict())
print('Setelah SMOTE  -- train:', X_train_res.shape, '| distribusi kelas:', y_train_res.value_counts().to_dict())
print()
print('Test TIDAK di-resample, tetap:', X_test.shape, '| distribusi kelas:', y_test.value_counts().to_dict())


Sebelum SMOTE -- train: (8631, 20) | distribusi kelas: {0: 7295, 1: 1336}
Setelah SMOTE  -- train: (14590, 20) | distribusi kelas: {0: 7295, 1: 7295}

Test TIDAK di-resample, tetap: (3699, 20) | distribusi kelas: {0: 3127, 1: 572}


## 2. Training 5 Model (Parameter Sesuai Paper, Tanpa Tuning Ulang)

In [3]:
models = get_models()
results = {}
trained_models = {}

for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train_res, y_train_res)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    results[name] = evaluate_model(y_test, y_pred, y_proba)
    trained_models[name] = model
    print(f'  selesai. Accuracy={results[name]["Accuracy"]:.4f}, auROC={results[name]["auROC"]:.4f}')

print()
print('Semua model selesai dilatih dan dievaluasi.')


Training Decision Tree...
  selesai. Accuracy=0.8662, auROC=0.9091
Training SVM...


/Users/vickymahfudy/Study/Term 2/Data Science/Replikasi_Paper/.venv/lib/python3.11/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


  selesai. Accuracy=0.8667, auROC=0.8814
Training MLP...


  selesai. Accuracy=0.8543, auROC=0.9067
Training Random Forest...


  selesai. Accuracy=0.8827, auROC=0.9153
Training XGBoost...
  selesai. Accuracy=0.8867, auROC=0.9230

Semua model selesai dilatih dan dievaluasi.


## 3. Tabel Hasil Replikasi (8 Metrik, 5 Model)

In [4]:
results_df = pd.DataFrame(results).T
results_df = results_df[['Accuracy', 'Precision', 'F1-Score', 'TPR (Recall)', 'TNR', 'auROC', 'auPR', 'MCC']]
results_df = results_df.round(4)
results_df.to_csv('../results/tables/model_results.csv')
results_df


,Accuracy,Precision,F1-Score,TPR (Recall),TNR,auROC,auPR,MCC
Decision Tree,0.8662,0.5470,0.6441,0.7832,0.8814,0.9091,0.6453,0.5787
SVM,0.8667,0.5514,0.6324,0.7413,0.8897,0.8814,0.6132,0.5621
MLP,0.8543,0.5188,0.6280,0.7955,0.8650,0.9067,0.6460,0.5615
Random Forest,0.8827,0.6018,0.6528,0.7133,0.9137,0.9153,0.6775,0.5859
XGBoost,0.8867,0.6114,0.6672,0.7343,0.9146,0.9230,0.7051,0.6033


## 4. Perbandingan dengan Hasil Paper

Paper melaporkan hasil terbaik XGBoost (max_depth=2) dan Decision Tree (max_depth=5) pada kombinasi chi2+SMOTE. Nilai paper: XGBoost Accuracy=90,65%, Precision=90,01, F1=0,898, TPR=0,801, TNR=0,96, auROC=0,937, auPR=0,749, MCC=0,599. Decision Tree Accuracy=90,54%, Precision=89,9, F1=0,8998, TPR=0,76, TNR=0,93, auROC=0,923, auPR=0,731, MCC=0,605.

In [5]:
paper_results = {
    'XGBoost': {
        'Accuracy': 0.9065, 'Precision': 0.9001, 'F1-Score': 0.898, 'TPR (Recall)': 0.801,
        'TNR': 0.96, 'auROC': 0.937, 'auPR': 0.749, 'MCC': 0.599,
    },
    'Decision Tree': {
        'Accuracy': 0.9054, 'Precision': 0.899, 'F1-Score': 0.8998, 'TPR (Recall)': 0.76,
        'TNR': 0.93, 'auROC': 0.923, 'auPR': 0.731, 'MCC': 0.605,
    },
}

comparison_rows = []
metrics = ['Accuracy', 'Precision', 'F1-Score', 'TPR (Recall)', 'TNR', 'auROC', 'auPR', 'MCC']
for model_name, paper_vals in paper_results.items():
    for metric in metrics:
        paper_val = paper_vals[metric]
        repl_val = results_df.loc[model_name, metric]
        diff = repl_val - paper_val
        comparison_rows.append({
            'Model': model_name,
            'Metric': metric,
            'Paper Result': paper_val,
            'Replication Result': round(repl_val, 4),
            'Difference': round(diff, 4),
        })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv('../results/tables/comparison_paper_vs_replication.csv', index=False)
comparison_df


,Model,Metric,Paper Result,Replication Result,Difference
0,XGBoost,Accuracy,0.9065,0.8867,-0.0198
1,XGBoost,Precision,0.9001,0.6114,-0.2887
2,XGBoost,F1-Score,0.8980,0.6672,-0.2308
3,XGBoost,TPR (Recall),0.8010,0.7343,-0.0667
4,XGBoost,TNR,0.9600,0.9146,-0.0454
5,XGBoost,auROC,0.9370,0.9230,-0.0140
6,XGBoost,auPR,0.7490,0.7051,-0.0439
7,XGBoost,MCC,0.5990,0.6033,0.0043
8,Decision Tree,Accuracy,0.9054,0.8662,-0.0392
9,Decision Tree,Precision,0.8990,0.5470,-0.3520


### Investigasi Selisih Precision dan F1

Selisih Accuracy/auROC/auPR/MCC relatif kecil (1-5 poin), tapi Precision dan F1 meleset jauh (23-35 poin) di kedua model. Pola ini sistematis, jadi perlu dicari akar masalahnya sebelum langsung disimpulkan replikasi gagal.

In [6]:
from sklearn.metrics import classification_report, precision_score, f1_score

best_model = trained_models['XGBoost']
y_pred_xgb = best_model.predict(X_test)

print('Classification report XGBoost (per kelas):')
print(classification_report(y_test, y_pred_xgb, digits=4))

print('Precision (kelas positif saja, default sklearn):', round(precision_score(y_test, y_pred_xgb), 4))
print('Precision (weighted average, kedua kelas):      ', round(precision_score(y_test, y_pred_xgb, average='weighted'), 4))
print('F1 (kelas positif saja, default sklearn):        ', round(f1_score(y_test, y_pred_xgb), 4))
print('F1 (weighted average, kedua kelas):              ', round(f1_score(y_test, y_pred_xgb, average='weighted'), 4))


Classification report XGBoost (per kelas):
              precision    recall  f1-score   support

           0     0.9495    0.9146    0.9317      3127
           1     0.6114    0.7343    0.6672       572

    accuracy                         0.8867      3699
   macro avg     0.7804    0.8244    0.7995      3699
weighted avg     0.8972    0.8867    0.8908      3699

Precision (kelas positif saja, default sklearn): 0.6114
Precision (weighted average, kedua kelas):       0.8972
F1 (kelas positif saja, default sklearn):         0.6672
F1 (weighted average, kedua kelas):               0.8908


**Temuan:** kalau Precision dan F1 dihitung sebagai weighted average (bukan kelas positif saja), nilainya jadi sangat dekat dengan paper (selisih <1 poin). Ini nunjukin paper kemungkinan besar melaporkan weighted average, bukan skor kelas positif murni, meski nggak dijelaskan eksplisit di papernya.

Tabel hasil utama tetap pakai Precision dan F1 kelas positif saja, karena buat masalah prediksi niat beli yang imbalanced, metrik kelas positif jauh lebih relevan secara bisnis. Selisih besar di tabel perbandingan jadi soal perbedaan definisi metrik, bukan kegagalan pipeline.

## 5. Diskusi Perbandingan Hasil

**Apakah performa paper berhasil direproduksi?**
Untuk metrik yang tidak bergantung pada cara averaging kelas (Accuracy, auROC, auPR, MCC), hasilnya cukup dekat: selisih sekitar 1-9 poin persentase, bahkan MCC XGBoost cuma berselisih 0,004. Mengingat perbedaan random seed, versi library, dan detail preprocessing yang nggak seluruhnya didokumentasikan paper, ini dianggap cukup untuk bilang performa inti paper berhasil direproduksi.

Selisih besar di Precision/F1 bukan tanda replikasi gagal, cuma beda definisi metrik seperti dijelaskan di atas.

**Kemungkinan penyebab selisih yang tersisa:**
- Random seed dan komposisi persis train/test yang pasti berbeda dari paper (paper nggak sebut random_state)
- Versi library yang berbeda (scikit-learn, XGBoost, imbalanced-learn saat ini vs 2023)
- Koreksi metodologi (fit hanya di train) yang secara konsep bikin estimasi performa sedikit lebih konservatif, ketimbang kalau paper punya leakage ringan
- MLP/SVM yang cukup sensitif terhadap random initialization

**Apakah koreksi metodologi memengaruhi hasil?**
Untuk mengisolasi murni efek koreksi ini butuh eksperimen pembanding tanpa koreksi, yang di luar scope Tugas #2. Tapi secara konsep, arah selisih hasil replikasi (semua metrik sedikit lebih rendah dari paper, bukan lebih tinggi) konsisten dengan hipotesis bahwa paper mungkin punya leakage ringan yang menggelembungkan performa mereka.

**Limitasi:**
- Definisi Precision/F1 paper nggak bisa dipastikan (kelas positif vs weighted average)
- Hanya skenario kombinasi terbaik yang direplikasi, bukan seluruh ablation study
- Detail proses tuning hyperparameter paper nggak didokumentasikan, jadi cuma nilai parameter akhirnya yang dipakai
- SVM dan MLP sensitif terhadap random initialization, hasil single-run mungkin nggak mewakili variasi across multiple run

## 6. Ringkasan File Output

In [7]:
print('Modeling dan evaluasi selesai. File tersimpan:')
print('- results/tables/model_results.csv')
print('- results/tables/comparison_paper_vs_replication.csv')


Modeling dan evaluasi selesai. File tersimpan:
- results/tables/model_results.csv
- results/tables/comparison_paper_vs_replication.csv
